In [ ]:
# [Cell 0]
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/football-tracker")
else:
    DRIVE = Path("drive_checkpoints")
DRIVE.mkdir(parents=True, exist_ok=True)

MODELS_DIR = DRIVE / "models"
MODELS_DIR.mkdir(exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
print(f"Checkpoints: {DRIVE}")
print(f"Models: {MODELS_DIR}")

In [ ]:
# [Cell 1]
!pip install -q ultralytics huggingface_hub
from ultralytics import YOLO, settings
settings.update({"runs_dir": str(DRIVE / "runs")})
import yaml
print("Ultralytics ready.")

In [ ]:
# [Cell 2]
from huggingface_hub import snapshot_download

dataset_path = DATA_DIR / "football_pitch"
snapshot_download(
    "martinjolif/football-pitch-detection",
    repo_type="dataset",
    local_dir=str(dataset_path),
)
print(f"Downloaded to {dataset_path}")
print("Files:", list(dataset_path.iterdir()))

In [ ]:
# [Cell 3]
data_yaml = dataset_path / "data.yaml"
if not data_yaml.exists():
    kps = [[x, y, 2] for _ in range(32) for x, y in [(0, 0)]]
    cfg = {
        "path": str(dataset_path.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "kpt_shape": [32, 3],
        "nc": 0,
        "names": [],
    }
    yaml.dump(cfg, open(data_yaml, "w"))
    print(f"Created {data_yaml}")
else:
    print(f"Using existing {data_yaml}")

In [ ]:
# [Cell 4]
model = YOLO("yolo26s-pose.pt")
results = model.train(
    data=str(data_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    name="football_pitch",
    pretrained=True,
    device=0,
    save_period=10,
    degrees=5,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)
print(f"Best model: {results.save_dir}")

In [ ]:
# [Cell 5]
runs = sorted((DRIVE / "runs" / "pose").glob("football_pitch*"), reverse=True)
best_pt = runs[0] / "weights" / "best.pt"
model = YOLO(str(best_pt))
model.export(format="onnx")
print(f"Exported ONNX from {best_pt}")

In [ ]:
# [Cell 6]
import shutil

runs = sorted((DRIVE / "runs" / "pose").glob("football_pitch*"), reverse=True)
src_pt = runs[0] / "weights" / "best.pt"
src_onnx = Path(str(src_pt).replace(".pt", ".onnx"))

shutil.copy(src_pt, MODELS_DIR / "pitch_detector.pt")
if src_onnx.exists():
    shutil.copy(src_onnx, MODELS_DIR / "pitch_detector.onnx")

for f in MODELS_DIR.iterdir():
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")
print("Saved to Drive.")